In [1]:
import numpy as np
import openmdao.api as om
import os

os.environ['OPENMDAO_REPORTS'] = 'none'

[G3FM726NKV:62281] shmem: mmap: an error occurred while determining whether or not /var/folders/m7/3rglclls3gs685x7cpg2fpt0hwg1yd/T//ompi.G3FM726NKV.566724557/jf.0/3407806464/sm_segment.G3FM726NKV.566724557.cb1f0000.0 could be created.


In [3]:
class satelliteDis1(om.ExplicitComponent):
    def setup(self):
        # Global Design Variable
        self.add_input('x', val=np.ones(5))
        
        # Coupling parameter
        self.add_input('u21', val=9.)

        # Coupling output
        self.add_output('u12', val=9.)

        self.counter = 0

    def setup_partials(self):
        # Finite difference all partials
        self.declare_partials('*', '*', method='cs')

    def compute(self, inputs, outputs):
        x1 = inputs['x'][0]
        x2 = inputs['x'][1]
        x3 = inputs['x'][2]
        u21 = inputs['u21']
        self.counter += 1
        # print('dis1')

        if u21.real < 0.0:
            u21 *= -1

        outputs['u12'] = x1**2 + 2*x2 - x3 + 2*(u21**0.5)

class satelliteDis2(om.ExplicitComponent):
    def setup(self):
        # Global Design Variable
        self.add_input('x', val=np.ones(5))
        
        # Coupling parameter
        self.add_input('u12', val=9.)

        # Coupling output
        self.add_output('u21', val=9.)

        self.counter = 0
        
    def setup_partials(self):
        # Finite difference all partials
        self.declare_partials('*', '*', method='cs')

    def compute(self, inputs, outputs):
        x1 = inputs['x'][0]
        x4 = inputs['x'][3]
        x5 = inputs['x'][4]
        u12 = inputs['u12']
        self.counter += 1
        # print('dis2')

        outputs['u21'] = x1*x4 + x4**2 + x5 + u12

In [5]:
class satelliteGroup(om.Group):
    def setup(self):
        cycle = self.add_subsystem('cycle', om.Group(), promotes=['*'])
        cycle.add_subsystem('d1', satelliteDis1(), promotes_inputs=['x', 'u21'], 
                           promotes_outputs=['u12'])
        cycle.add_subsystem('d2', satelliteDis2(), promotes_inputs=['x', 'u12'], 
                           promotes_outputs=['u21'])

        nlbgs = cycle.nonlinear_solver = om.NonlinearBlockGS()
        # nlbgs = cycle.nonlinear_solver = om.NewtonSolver(solve_subsystems=True, iprint=2)
        nlbgs.options['maxiter'] = 1000
        # nlbgs.options['iprint'] = 2

        self.add_subsystem('obj_cmp', om.ExecComp('f = ((x[0]**0.5 + x[3] + 0.4*x[0]*x[4]) - (4.5 - (x[0]**2 + 2*x[1] + x[2] + x[1]*exp(-u21))))', x=np.ones(5), u21=9.0),
                           promotes_inputs=['x','u21'], promotes_outputs=['f'])
        self.add_subsystem('res1', om.ExecComp('r1 = u12 - (x[0]**2 + 2*x[1] - x[2] + 2*(u21**0.5))', x=np.ones(5)*0.5, u21=9.0),
                           promotes_inputs=['x','u21'], promotes_outputs=['r1'])
        self.add_subsystem('res2', om.ExecComp('r2 = u21 - (x[0]*x[3] + x[3]**2 + x[4] + u12)', x=np.ones(5), u12=9.0),
                           promotes_inputs=['x','u12'], promotes_outputs=['r2'])    

In [7]:
prob = om.Problem()
prob.model = satelliteGroup()

prob.driver = om.ScipyOptimizeDriver()
prob.driver.options['optimizer'] = 'COBYQA'
prob.driver.options['tol'] = 1e-8
prob.driver.options['disp'] = False

prob.model.add_design_var('x', lower=np.ones(5)*0, upper=np.ones(5)*2)
prob.model.add_objective('f')
prob.model.add_constraint('r1', lower=0., upper=0.)
prob.model.add_constraint('r2', lower=0., upper=0.)
# prob.model.add_constraint('u21', lower=0.)
prob.model.add_constraint('u12', lower=0.)

prob.model.set_input_defaults('x', np.ones(5))

prob.model.approx_totals()

prob.setup()

prob.set_val('x',np.ones(5))

prob.run_model()

# prob.run_driver()

print('min objective', prob.get_val('f'))

print('residuals')
print('r1', prob.get_val('r1'))
print('r2', prob.get_val('r2'))

print('design vars')
print('x', prob.get_val('x'))

print('coupling vars')
print('u12', prob.get_val('u12'))
print('u21', prob.get_val('u21'))


=====
cycle
=====
NL: NLBGS Converged in 20 iterations
min objective [1.9000068]
residuals
r1 [-7.89897949]
r2 [-10.89897949]
design vars
x [1. 1. 1. 1. 1.]
coupling vars
u12 [8.89897949]
u21 [11.89897949]


In [9]:
prob.model.cycle.d1.counter

20

In [11]:
prob.model.cycle.d2.counter

20